# Proyecto Final: Calculadoras de Instrumentos de Deuda y Construccion de Curvas


## Contenido
1. Utilidades y Validaciones
2. Ejercicio 1: Calculadora de CETE
3. Ejercicio 2: Calculadora de M-BONO
4. Ejercicio 3: Calculadora de UDIBONO
5. Ejercicio 4: Construccion de Curvas (Bootstrapping y Forwards)
6. Menu Interactivo - Calculadora
7. Demostraciones

## 1. Utilidades y Validaciones

In [ ]:
from datetime import datetime, timedelta
import numpy as np
from scipy.optimize import fsolve


def validar_fecha(fecha, nombre_campo="fecha"):
    """Valida y convierte una fecha al formato datetime."""
    if isinstance(fecha, datetime):
        return fecha

    if not isinstance(fecha, str):
        print("ERROR: {} debe ser string o datetime".format(nombre_campo))
        return None

    try:
        return datetime.strptime(fecha, "%Y-%m-%d")
    except ValueError:
        try:
            return datetime.strptime(fecha, "%d/%m/%Y")
        except ValueError:
            print("ERROR: {} formato invalido. Use YYYY-MM-DD o DD/MM/YYYY".format(nombre_campo))
            return None


def validar_numero(valor, nombre_campo="valor", minimo=None, maximo=None):
    """Valida que un valor sea numerico y este dentro de rangos opcionales."""
    try:
        valor_num = float(valor)
    except (TypeError, ValueError):
        print("ERROR: {} debe ser numerico".format(nombre_campo))
        return None

    if not np.isfinite(valor_num):
        print("ERROR: {} debe ser finito".format(nombre_campo))
        return None

    if minimo is not None and valor_num < minimo:
        print("ERROR: {} debe ser >= {}".format(nombre_campo, minimo))
        return None

    if maximo is not None and valor_num > maximo:
        print("ERROR: {} debe ser <= {}".format(nombre_campo, maximo))
        return None

    return valor_num


def validar_fechas_coherentes(fecha_valuacion, fecha_vencimiento):
    """Valida que la fecha de vencimiento sea posterior a la fecha de valuacion."""
    if fecha_vencimiento <= fecha_valuacion:
        print("ERROR: Fecha vencimiento debe ser posterior a fecha valuacion")
        return False
    return True


def obtener_feriados_mexico(anio):
    """
    Retorna lista de dias feriados oficiales en Mexico para un anio.

    Incluye feriados fijos y variables segun la Ley Federal del Trabajo:
    - 1 de enero: Anio Nuevo
    - Primer lunes de febrero: Dia de la Constitucion
    - Tercer lunes de marzo: Natalicio de Benito Juarez
    - 1 de mayo: Dia del Trabajo
    - 16 de septiembre: Independencia de Mexico
    - Tercer lunes de noviembre: Revolucion Mexicana
    - 25 de diciembre: Navidad
    """
    feriados = [
        datetime(anio, 1, 1),   # Anio Nuevo
        datetime(anio, 5, 1),   # Dia del Trabajo
        datetime(anio, 9, 16),  # Independencia de Mexico
        datetime(anio, 12, 25), # Navidad
    ]

    # Primer lunes de febrero (Dia de la Constitucion)
    fecha = datetime(anio, 2, 1)
    while fecha.weekday() != 0:
        fecha += timedelta(days=1)
    feriados.append(fecha)

    # Tercer lunes de marzo (Natalicio de Benito Juarez)
    fecha = datetime(anio, 3, 1)
    lunes_count = 0
    while lunes_count < 3:
        if fecha.weekday() == 0:
            lunes_count += 1
        if lunes_count < 3:
            fecha += timedelta(days=1)
    feriados.append(fecha)

    # Tercer lunes de noviembre (Revolucion Mexicana)
    fecha = datetime(anio, 11, 1)
    lunes_count = 0
    while lunes_count < 3:
        if fecha.weekday() == 0:
            lunes_count += 1
        if lunes_count < 3:
            fecha += timedelta(days=1)
    feriados.append(fecha)

    return feriados


def es_dia_laboral(fecha):
    """
    Determina si una fecha es dia laboral en Mexico.

    No es dia laboral si:
    - Es fin de semana (sabado o domingo)
    - Es dia feriado oficial en Mexico
    """
    if fecha.weekday() >= 5:
        return False

    feriados = obtener_feriados_mexico(fecha.year)
    return fecha.date() not in [f.date() for f in feriados]


def ajustar_dia_laboral(fecha, convencion="siguiente"):
    """
    Ajusta una fecha al dia laboral mas cercano segun convencion.

    Args:
        fecha: Fecha a ajustar
        convencion: "siguiente", "anterior" o "modificado_siguiente"

    Returns:
        Fecha ajustada al dia laboral mas cercano
    """
    if es_dia_laboral(fecha):
        return fecha

    if convencion == "siguiente":
        while not es_dia_laboral(fecha):
            fecha += timedelta(days=1)
        return fecha
    elif convencion == "anterior":
        while not es_dia_laboral(fecha):
            fecha -= timedelta(days=1)
        return fecha
    elif convencion == "modificado_siguiente":
        fecha_original = fecha
        while not es_dia_laboral(fecha):
            fecha += timedelta(days=1)
        if fecha.month != fecha_original.month:
            fecha = fecha_original
            while not es_dia_laboral(fecha):
                fecha -= timedelta(days=1)
        return fecha
    else:
        print("ERROR: Convencion '{}' invalida".format(convencion))
        return fecha


def calcular_dias_entre_fechas(fecha_inicio, fecha_fin, base_dias):
    """Calcula fraccion de anio entre fechas segun convencion de conteo."""
    if base_dias.upper() in ["ACTUAL/360", "ACT/360"]:
        dias = (fecha_fin - fecha_inicio).days
        return dias / 360.0
    elif base_dias.upper() in ["ACTUAL/365", "ACT/365"]:
        dias = (fecha_fin - fecha_inicio).days
        return dias / 365.0
    elif base_dias.upper() == "30/360":
        d1, m1, y1 = fecha_inicio.day, fecha_inicio.month, fecha_inicio.year
        d2, m2, y2 = fecha_fin.day, fecha_fin.month, fecha_fin.year
        if d1 == 31:
            d1 = 30
        if d2 == 31 and d1 >= 30:
            d2 = 30
        dias = 360 * (y2 - y1) + 30 * (m2 - m1) + (d2 - d1)
        return dias / 360.0
    else:
        print("ERROR: Base de dias '{}' invalida".format(base_dias))
        return None


def generar_fechas_cupon(fecha_inicio, fecha_vencimiento, frecuencia):
    """Genera fechas de pago de cupones."""
    if frecuencia not in [1, 2, 4, 12]:
        print("ERROR: Frecuencia {} invalida".format(frecuencia))
        return None

    meses_entre_cupones = 12 // frecuencia
    fechas = []
    fecha_actual = fecha_vencimiento

    while fecha_actual > fecha_inicio:
        fechas.append(fecha_actual)
        anio = fecha_actual.year
        mes = fecha_actual.month - meses_entre_cupones

        if mes <= 0:
            mes += 12
            anio -= 1

        try:
            fecha_actual = fecha_actual.replace(year=anio, month=mes)
        except ValueError:
            if mes == 2:
                dia = 28 if anio % 4 != 0 or (anio % 100 == 0 and anio % 400 != 0) else 29
            else:
                dias_en_mes = [31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]
                dia = dias_en_mes[mes - 1]
            fecha_actual = datetime(anio, mes, dia)

    return sorted(fechas)


def interpolar_lineal(x, x0, x1, y0, y1):
    """Interpolacion lineal entre dos puntos."""
    if x1 == x0:
        return y0
    return y0 + (y1 - y0) * (x - x0) / (x1 - x0)

### Prueba de validacion de feriados

In [ ]:
# Demostrar que detecta feriados
print("Validacion de dias feriados en Mexico 2025:")
print()

feriados_2025 = obtener_feriados_mexico(2025)
dias_semana = ['lunes', 'martes', 'miercoles', 'jueves', 'viernes', 'sabado', 'domingo']
for feriado in sorted(feriados_2025):
    dia_semana = dias_semana[feriado.weekday()]
    print("{} ({}): No laboral".format(feriado.strftime('%Y-%m-%d'), dia_semana))

print()
print("Ejemplos de validacion:")
fecha_test = datetime(2025, 1, 1)
print("2025-01-01 (Anio Nuevo - miercoles): es_dia_laboral = {}".format(es_dia_laboral(fecha_test)))
dia_ajustado = ajustar_dia_laboral(fecha_test)
print("  Ajustado: {} ({})".format(dia_ajustado.strftime('%Y-%m-%d'), dias_semana[dia_ajustado.weekday()]))

fecha_test = datetime(2025, 1, 2)
print("2025-01-02 (jueves normal): es_dia_laboral = {}".format(es_dia_laboral(fecha_test)))

Validacion de dias feriados en Mexico 2025:

2025-01-01 (miercoles): No laboral
2025-02-03 (lunes): No laboral
2025-03-17 (lunes): No laboral
2025-05-01 (jueves): No laboral
2025-09-16 (martes): No laboral
2025-11-17 (lunes): No laboral
2025-12-25 (jueves): No laboral

Ejemplos de validacion:
2025-01-01 (Anio Nuevo - miercoles): es_dia_laboral = False
  Ajustado: 2025-01-02 (jueves)
2025-01-02 (jueves normal): es_dia_laboral = True


## 2. Ejercicio 1: Calculadora de CETE

In [ ]:
def calcular_precio_cete(fecha_valuacion, fecha_vencimiento, tasa_descuento, base_dias="Actual/360"):
    """
    Calcula el precio de un CETE.

    Args:
        fecha_valuacion: Fecha de valuacion
        fecha_vencimiento: Fecha de vencimiento
        tasa_descuento: Tasa de descuento anualizada (decimal)
        base_dias: Convencion de conteo de dias

    Returns:
        Diccionario con resultados de valuacion o None si hay error
    """
    VALOR_NOMINAL = 10.0

    # Validaciones
    fecha_val = validar_fecha(fecha_valuacion, "fecha_valuacion") # Validamos fecha de valuación
    if fecha_val is None:
        return None

    fecha_venc = validar_fecha(fecha_vencimiento, "fecha_vencimiento") # Validamos fecha de vencimiento
    if fecha_venc is None:
        return None

    tasa = validar_numero(tasa_descuento, "tasa_descuento", 0.0, 1.0) # Validamos la tasa de descuento
    if tasa is None:
        return None

    if not validar_fechas_coherentes(fecha_val, fecha_venc): # Revisamos que valuación < vencimiento
        return None

    # Ajuste de dias laborales
    fecha_val = ajustar_dia_laboral(fecha_val, "modificado_siguiente") # Ajustamos fecha de valuación a día hábil
    fecha_venc = ajustar_dia_laboral(fecha_venc, "modificado_siguiente")

    # Calculos
    dias_vencimiento = (fecha_venc - fecha_val).days
    fraccion_anio = calcular_dias_entre_fechas(fecha_val, fecha_venc, base_dias) # Calculamos fracción del año
    if fraccion_anio is None:
        return None

    precio = VALOR_NOMINAL / (1 + tasa * fraccion_anio)
    precio = round(precio, 7)# Redondeamos a 7 decimales

    return {
        "instrumento": "CETE",
        "fecha_valuacion": fecha_val.strftime("%Y-%m-%d"),
        "fecha_vencimiento": fecha_venc.strftime("%Y-%m-%d"),
        "dias_al_vencimiento": dias_vencimiento,
        "fraccion_anio": round(fraccion_anio, 6),
        "tasa_descuento": tasa,
        "base_dias": base_dias,
        "valor_nominal": VALOR_NOMINAL,
        "precio": precio,
        "descuento": round(VALOR_NOMINAL - precio, 7)
    }


def calcular_rendimiento_cete(fecha_valuacion, fecha_vencimiento, precio, base_dias="Actual/360"):
    """
    Calcula la tasa de rendimiento de un CETE dado su precio.

    Args:
        fecha_valuacion: Fecha de valuacion
        fecha_vencimiento: Fecha de vencimiento
        precio: Precio de mercado del CETE
        base_dias: Convencion de conteo de dias

    Returns:
        Tasa de rendimiento anualizada o None si hay error
    """
    VALOR_NOMINAL = 10.0

    # Validaciones
    fecha_val = validar_fecha(fecha_valuacion, "fecha_valuacion")
    if fecha_val is None:
        return None

    fecha_venc = validar_fecha(fecha_vencimiento, "fecha_vencimiento")
    if fecha_venc is None:
        return None

    precio_val = validar_numero(precio, "precio", 0.0, VALOR_NOMINAL)
    if precio_val is None:
        return None

    fraccion_anio = calcular_dias_entre_fechas(fecha_val, fecha_venc, base_dias)
    if fraccion_anio is None:
        return None

    if fraccion_anio == 0:
        print("ERROR: No se puede calcular rendimiento para fraccion de anio = 0")
        return None

    rendimiento = ((VALOR_NOMINAL / precio_val) - 1) / fraccion_anio
    return rendimiento

## 3. Ejercicio 2: Calculadora de M-BONO

In [ ]:
def calcular_mbono(fecha_valuacion, fecha_vencimiento, tasa_cupon, ytm, base_dias="Actual/365"):
    """
    Calcula precio y duraciones de un M-BONO.

    Args:
        fecha_valuacion: Fecha de valuacion
        fecha_vencimiento: Fecha de vencimiento
        tasa_cupon: Tasa cupon anual (decimal)
        ytm: Yield to maturity (decimal)
        base_dias: Convencion de conteo de dias

    Returns:
        Diccionario con resultados de valuacion o None si hay error
    """
    VALOR_NOMINAL = 100.0
    FRECUENCIA_CUPON = 2

    # Validaciones
    fecha_val = validar_fecha(fecha_valuacion, "fecha_valuacion")
    if fecha_val is None:
        return None

    fecha_venc = validar_fecha(fecha_vencimiento, "fecha_vencimiento")
    if fecha_venc is None:
        return None

    cupon = validar_numero(tasa_cupon, "tasa_cupon", 0.0, 1.0)
    if cupon is None:
        return None

    ytm_val = validar_numero(ytm, "ytm", -0.1, 1.0)
    if ytm_val is None:
        return None

    if not validar_fechas_coherentes(fecha_val, fecha_venc):
        return None

    # Ajuste de dias laborales
    fecha_val = ajustar_dia_laboral(fecha_val, "modificado_siguiente")
    fecha_venc = ajustar_dia_laboral(fecha_venc, "modificado_siguiente")

    # Generar fechas de cupon
    fechas_cupon = generar_fechas_cupon(fecha_val, fecha_venc, FRECUENCIA_CUPON)
    if fechas_cupon is None:
        return None

    cupon_periodico = VALOR_NOMINAL * cupon / FRECUENCIA_CUPON

    # Calcular flujos
    flujos = [cupon_periodico] * len(fechas_cupon)
    flujos[-1] += VALOR_NOMINAL

    precio_sucio = 0.0
    suma_ponderada_duracion = 0.0

    for fecha_flujo, flujo in zip(fechas_cupon, flujos):
        tiempo = calcular_dias_entre_fechas(fecha_val, fecha_flujo, base_dias)
        if tiempo is None:
            return None
        factor_descuento = (1 + ytm_val / FRECUENCIA_CUPON) ** (tiempo * FRECUENCIA_CUPON)
        valor_presente = flujo / factor_descuento
        precio_sucio += valor_presente
        suma_ponderada_duracion += tiempo * valor_presente

    # Calcular interes devengado
    fecha_ultimo_cupon = None
    for fecha in reversed(fechas_cupon):
        if fecha <= fecha_val:
            fecha_ultimo_cupon = fecha
            break

    if fecha_ultimo_cupon is None:
        primer_cupon = fechas_cupon[0]
        anio = primer_cupon.year
        mes = primer_cupon.month - 6
        if mes <= 0:
            mes += 12
            anio -= 1
        fecha_ultimo_cupon = datetime(anio, mes, primer_cupon.day)

    fecha_proximo_cupon = fechas_cupon[0]
    for fecha in fechas_cupon:
        if fecha > fecha_val:
            fecha_proximo_cupon = fecha
            break

    dias_transcurridos = (fecha_val - fecha_ultimo_cupon).days
    dias_periodo = (fecha_proximo_cupon - fecha_ultimo_cupon).days

    if dias_periodo == 0:
        interes_devengado = 0.0
    else:
        fraccion = dias_transcurridos / dias_periodo
        interes_devengado = cupon_periodico * fraccion

    precio_limpio = precio_sucio - interes_devengado

    # Calcular duraciones
    duracion_macaulay = suma_ponderada_duracion / precio_sucio if precio_sucio > 0 else 0.0
    duracion_modificada = duracion_macaulay / (1 + ytm_val / FRECUENCIA_CUPON)

    return {
        "instrumento": "M-BONO",
        "fecha_valuacion": fecha_val.strftime("%Y-%m-%d"),
        "fecha_vencimiento": fecha_venc.strftime("%Y-%m-%d"),
        "tasa_cupon": cupon,
        "ytm": ytm_val,
        "base_dias": base_dias,
        "valor_nominal": VALOR_NOMINAL,
        "cupon_periodico": round(cupon_periodico, 4),
        "num_cupones_restantes": len(fechas_cupon),
        "precio_sucio": round(precio_sucio, 6),
        "interes_devengado": round(interes_devengado, 6),
        "precio_limpio": round(precio_limpio, 6),
        "duracion_macaulay": round(duracion_macaulay, 4),
        "duracion_modificada": round(duracion_modificada, 4)
    }

## 4. Ejercicio 3: Calculadora de UDIBONO

In [ ]:


def calcular_udibono(fecha_valuacion, fecha_vencimiento, tasa_cupon_real, valor_udi, ytm_real, base_dias="Actual/365"):
    """
    Calcula precio y duraciones de un UDIBONO.

    Args:
        fecha_valuacion: Fecha de valuacion
        fecha_vencimiento: Fecha de vencimiento
        tasa_cupon_real: Tasa cupon real anual en UDIs (decimal)
        valor_udi: Valor de la UDI en la fecha de valuacion
        ytm_real: Yield to maturity (decimal)
        base_dias: Convencion de conteo de dias

    Returns:
        Diccionario con resultados de valuacion o None si hay error
    """
    VALOR_NOMINAL_UDI = 100.0
    FRECUENCIA_CUPON = 2

    # Validaciones
    fecha_val = validar_fecha(fecha_valuacion, "fecha_valuacion")
    if fecha_val is None:
        return None

    fecha_venc = validar_fecha(fecha_vencimiento, "fecha_vencimiento")
    if fecha_venc is None:
        return None

    cupon_real = validar_numero(tasa_cupon_real, "tasa_cupon_real", 0.0, 0.5)
    if cupon_real is None:
        return None

    udi = validar_numero(valor_udi, "valor_udi", 0.0)
    if udi is None:
        return None

    ytm_r = validar_numero(ytm_real, "ytm_real", -0.1, 0.5)
    if ytm_r is None:
        return None

    if not validar_fechas_coherentes(fecha_val, fecha_venc):
        return None

    # Ajuste de dias laborales
    fecha_val = ajustar_dia_laboral(fecha_val, "modificado_siguiente")
    fecha_venc = ajustar_dia_laboral(fecha_venc, "modificado_siguiente")

    # Generar fechas de cupon
    fechas_cupon = generar_fechas_cupon(fecha_val, fecha_venc, FRECUENCIA_CUPON)
    if fechas_cupon is None:
        return None

    cupon_periodico_udi = VALOR_NOMINAL_UDI * cupon_real / FRECUENCIA_CUPON

    # Calcular flujos
    flujos_udi = [cupon_periodico_udi] * len(fechas_cupon)
    flujos_udi[-1] += VALOR_NOMINAL_UDI

    precio_sucio_udi = 0.0
    suma_ponderada_duracion = 0.0

    for fecha_flujo, flujo_udi in zip(fechas_cupon, flujos_udi):
        tiempo = calcular_dias_entre_fechas(fecha_val, fecha_flujo, base_dias)
        if tiempo is None:
            return None
        factor_descuento = (1 + ytm_r / FRECUENCIA_CUPON) ** (tiempo * FRECUENCIA_CUPON)
        valor_presente = flujo_udi / factor_descuento
        precio_sucio_udi += valor_presente
        suma_ponderada_duracion += tiempo * valor_presente

    # Calcular interes devengado
    fecha_ultimo_cupon = None
    for fecha in reversed(fechas_cupon):
        if fecha <= fecha_val:
            fecha_ultimo_cupon = fecha
            break

    if fecha_ultimo_cupon is None:
        primer_cupon = fechas_cupon[0]
        anio = primer_cupon.year
        mes = primer_cupon.month - 6
        if mes <= 0:
            mes += 12
            anio -= 1
        fecha_ultimo_cupon = datetime(anio, mes, primer_cupon.day)

    fecha_proximo_cupon = fechas_cupon[0]
    for fecha in fechas_cupon:
        if fecha > fecha_val:
            fecha_proximo_cupon = fecha
            break

    dias_transcurridos = (fecha_val - fecha_ultimo_cupon).days
    dias_periodo = (fecha_proximo_cupon - fecha_ultimo_cupon).days

    if dias_periodo == 0:
        interes_devengado_udi = 0.0
    else:
        fraccion = dias_transcurridos / dias_periodo
        interes_devengado_udi = cupon_periodico_udi * fraccion

    precio_limpio_udi = precio_sucio_udi - interes_devengado_udi

    # Calcular duraciones
    duracion_macaulay = suma_ponderada_duracion / precio_sucio_udi if precio_sucio_udi > 0 else 0.0
    duracion_modificada = duracion_macaulay / (1 + ytm_r / FRECUENCIA_CUPON)

    return {
        "instrumento": "UDIBONO",
        "fecha_valuacion": fecha_val.strftime("%Y-%m-%d"),
        "fecha_vencimiento": fecha_venc.strftime("%Y-%m-%d"),
        "tasa_cupon_real": cupon_real,
        "ytm_real": ytm_r,
        "valor_udi": udi,
        "base_dias": base_dias,
        "valor_nominal_udi": VALOR_NOMINAL_UDI,
        "cupon_periodico_udi": round(cupon_periodico_udi, 4),
        "num_cupones_restantes": len(fechas_cupon),
        "precio_sucio_udi": round(precio_sucio_udi, 6),
        "interes_devengado_udi": round(interes_devengado_udi, 6),
        "precio_limpio_udi": round(precio_limpio_udi, 6),
        "precio_sucio_pesos": round(precio_sucio_udi * udi, 2),
        "interes_devengado_pesos": round(interes_devengado_udi * udi, 2),
        "precio_limpio_pesos": round(precio_limpio_udi * udi, 2),
        "duracion_macaulay": round(duracion_macaulay, 4),
        "duracion_modificada": round(duracion_modificada, 4)
    }

## 5. Ejercicio 4: Construccion de Curvas

In [ ]:
def construir_curva_cero(datos_mercado, frecuencia_cupon=2):
    """
    Calcula curva cupon cero mediante bootstrapping.

    Args:
        datos_mercado: Lista de tuplas (plazo_anios, tasa_cupon, ytm)
        frecuencia_cupon: Numero de pagos por año

    Returns:
        Diccionario {plazo: tasa_cero} o None si hay error
    """
    if not datos_mercado:
        print("ERROR: Se requiere al menos un punto de datos")
        return None

    # Validar datos
    datos_validados = []
    for plazo, cupon, ytm in datos_mercado:
        plazo_val = validar_numero(plazo, "plazo", 0.0, 100.0)
        cupon_val = validar_numero(cupon, "cupon", 0.0, 1.0)
        ytm_val = validar_numero(ytm, "ytm", -0.1, 1.0)

        if plazo_val is None or cupon_val is None or ytm_val is None:
            return None

        datos_validados.append((plazo_val, cupon_val, ytm_val))

    datos_validados.sort(key=lambda x: x[0])
    tasas_cero = {}

    def calcular_flujos_bono(plazo, tasa_cupon):
        periodos = int(plazo * frecuencia_cupon)
        dt = 1.0 / frecuencia_cupon
        cupon_periodico = tasa_cupon / frecuencia_cupon

        flujos = []
        for i in range(1, periodos + 1):
            tiempo = i * dt
            flujo = cupon_periodico
            if i == periodos:
                flujo += 1.0
            flujos.append((tiempo, flujo))

        return flujos

    def obtener_tasa_cero_interpolada(tiempo):
        tiempos_conocidos = sorted(tasas_cero.keys())

        if not tiempos_conocidos:
            print("ERROR: No hay tasas cero disponibles")
            return None

        if tiempo in tasas_cero:
            return tasas_cero[tiempo]

        if tiempo < tiempos_conocidos[0]:
            return tasas_cero[tiempos_conocidos[0]]

        if tiempo > tiempos_conocidos[-1]:
            return tasas_cero[tiempos_conocidos[-1]]

        for i in range(len(tiempos_conocidos) - 1):
            t0, t1 = tiempos_conocidos[i], tiempos_conocidos[i + 1]
            if t0 <= tiempo <= t1:
                return interpolar_lineal(tiempo, t0, t1, tasas_cero[t0], tasas_cero[t1])

        return tasas_cero[tiempos_conocidos[-1]]

    # Bootstrapping
    for plazo, tasa_cupon, ytm in datos_validados:
        if tasa_cupon == 0 or plazo <= 1.0 / frecuencia_cupon:
            tasas_cero[plazo] = ytm
            continue

        flujos = calcular_flujos_bono(plazo, tasa_cupon)
        precio_teorico = 1.0

        def ecuacion_precio(tasa_cero_objetivo):
            precio_calculado = 0.0

            for tiempo, flujo in flujos[:-1]:
                tasa_descuento = obtener_tasa_cero_interpolada(tiempo)
                if tasa_descuento is None:
                    tasa_descuento = tasa_cero_objetivo

                precio_calculado += flujo * np.exp(-tasa_descuento * tiempo)

            tiempo_final, flujo_final = flujos[-1]
            precio_calculado += flujo_final * np.exp(-tasa_cero_objetivo * tiempo_final)

            return precio_calculado - precio_teorico

        try:
            tasa_cero_solucion = fsolve(ecuacion_precio, ytm)[0]
            tasas_cero[plazo] = tasa_cero_solucion
        except:
            tasas_cero[plazo] = ytm

    return tasas_cero


def calcular_tasa_forward(tasas_cero, t1, t2):
    """
    Calcula tasa forward entre dos plazos.

    Formula: (1 + z_t2)^t2 = (1 + z_t1)^t1 x (1 + f_t1,t2)^(t2-t1)

    Args:
        tasas_cero: Diccionario con tasas cero
        t1: Plazo inicial
        t2: Plazo final

    Returns:
        Tasa forward o None si hay error
    """
    if t2 <= t1:
        print("ERROR: t2 ({}) debe ser mayor que t1 ({})".format(t2, t1))
        return None

    if not tasas_cero:
        print("ERROR: Se requieren tasas cero")
        return None

    tiempos_conocidos = sorted(tasas_cero.keys())

    def obtener_tasa(tiempo):
        if tiempo in tasas_cero:
            return tasas_cero[tiempo]

        for i in range(len(tiempos_conocidos) - 1):
            t0, t1_iter = tiempos_conocidos[i], tiempos_conocidos[i + 1]
            if t0 <= tiempo <= t1_iter:
                return interpolar_lineal(tiempo, t0, t1_iter, tasas_cero[t0], tasas_cero[t1_iter])

        if tiempo < tiempos_conocidos[0]:
            return tasas_cero[tiempos_conocidos[0]]
        return tasas_cero[tiempos_conocidos[-1]]

    z1 = obtener_tasa(t1)
    z2 = obtener_tasa(t2)

    tasa_forward = ((1 + z2) ** t2 / (1 + z1) ** t1) ** (1 / (t2 - t1)) - 1
    return tasa_forward


def construir_curvas_completas(datos_mercado, frecuencia_cupon=2):
    """
    Construye curva cero y calcula forwards.

    Args:
        datos_mercado: Lista de tuplas (plazo, cupon, ytm)
        frecuencia_cupon: Frecuencia de cupones

    Returns:
        Diccionario con curvas y forwards o None si hay error
    """
    tasas_cero = construir_curva_cero(datos_mercado, frecuencia_cupon)
    if tasas_cero is None:
        return None

    # Calcular forwards
    forward_1y1y = calcular_tasa_forward(tasas_cero, 1.0, 2.0)
    forward_1y2y = calcular_tasa_forward(tasas_cero, 1.0, 3.0)
    forward_1y3y = calcular_tasa_forward(tasas_cero, 1.0, 4.0)

    if forward_1y1y is None or forward_1y2y is None or forward_1y3y is None:
        return None

    forwards = {
        "1.0y1.0y": forward_1y1y,
        "1.0y2.0y": forward_1y2y,
        "1.0y3.0y": forward_1y3y,
    }

    return {
        "curva_cupon_original": [
            {"plazo": p, "cupon": c, "ytm": y} for p, c, y in datos_mercado
        ],
        "curva_cupon_cero": [
            {"plazo": p, "tasa_cero": round(t, 6)} for p, t in sorted(tasas_cero.items())
        ],
        "tasas_forward": {
            k: round(v, 6) for k, v in forwards.items()
        }
    }

## 6. Menu Interactivo - Calculadora

In [ ]:
def imprimir_resultado(resultado):
    """Imprime los resultados de forma ordenada."""
    if resultado is None:
        print("\n*** Ocurrio un error durante el calculo ***\n")
        return

    print("\n" + "="*60)
    print("RESULTADOS")
    print("="*60)
    for clave, valor in resultado.items():
        if isinstance(valor, float):
            print("{}: {:.6f}".format(clave, valor))
        else:
            print("{}: {}".format(clave, valor))
    print("="*60 + "\n")


def imprimir_curvas(resultado):
    """Imprime resultados de construccion de curvas."""
    if resultado is None:
        print("\n*** Ocurrio un error durante el calculo ***\n")
        return

    print("\n" + "="*60)
    print("CURVA CUPONADA ORIGINAL")
    print("="*60)
    for punto in resultado["curva_cupon_original"]:
        print("  Plazo={:.1f}, Cupon={:.4f}, YTM={:.4f}".format(
            punto['plazo'], punto['cupon'], punto['ytm']))

    print("\n" + "="*60)
    print("CURVA CUPON CERO")
    print("="*60)
    for punto in resultado["curva_cupon_cero"]:
        print("  Plazo={:.1f}, Tasa cero={:.6f}".format(
            punto['plazo'], punto['tasa_cero']))

    print("\n" + "="*60)
    print("TASAS FORWARD")
    print("="*60)
    for clave, tasa in resultado["tasas_forward"].items():
        print("  {}: {:.6f}".format(clave, tasa))
    print("="*60 + "\n")


def menu_cete():
    """Menu para calcular precio de CETE."""
    print("\n" + "="*60)
    print("CALCULADORA DE CETE")
    print("="*60)

    fecha_val = input("Fecha de valuacion (YYYY-MM-DD): ")
    fecha_venc = input("Fecha de vencimiento (YYYY-MM-DD): ")
    tasa = input("Tasa de descuento (decimal, ej: 0.0675): ")
    base = input("Base de dias (Actual/360 o Actual/365) [Actual/360]: ") or "Actual/360"

    try:
        tasa_float = float(tasa)
    except ValueError:
        print("ERROR: Tasa debe ser un numero")
        return

    resultado = calcular_precio_cete(fecha_val, fecha_venc, tasa_float, base)
    imprimir_resultado(resultado)


def menu_mbono():
    """Menu para calcular M-BONO."""
    print("\n" + "="*60)
    print("CALCULADORA DE M-BONO")
    print("="*60)

    fecha_val = input("Fecha de valuacion (YYYY-MM-DD): ")
    fecha_venc = input("Fecha de vencimiento (YYYY-MM-DD): ")
    tasa_cupon = input("Tasa cupon (decimal, ej: 0.07): ")
    ytm = input("YTM (decimal, ej: 0.0725): ")
    base = input("Base de dias (Actual/365) [Actual/365]: ") or "Actual/365"

    try:
        tasa_cupon_float = float(tasa_cupon)
        ytm_float = float(ytm)
    except ValueError:
        print("ERROR: Tasas deben ser numeros")
        return

    resultado = calcular_mbono(fecha_val, fecha_venc, tasa_cupon_float, ytm_float, base)
    imprimir_resultado(resultado)


def menu_udibono():
    """Menu para calcular UDIBONO."""
    print("\n" + "="*60)
    print("CALCULADORA DE UDIBONO")
    print("="*60)

    fecha_val = input("Fecha de valuacion (YYYY-MM-DD): ")
    fecha_venc = input("Fecha de vencimiento (YYYY-MM-DD): ")
    tasa_cupon = input("Tasa cupon real (decimal, ej: 0.04): ")
    valor_udi = input("Valor UDI (ej: 8.15): ")
    ytm = input("YTM real (decimal, ej: 0.042): ")
    base = input("Base de dias (Actual/365) [Actual/365]: ") or "Actual/365"

    try:
        tasa_cupon_float = float(tasa_cupon)
        valor_udi_float = float(valor_udi)
        ytm_float = float(ytm)
    except ValueError:
        print("ERROR: Valores deben ser numeros")
        return

    resultado = calcular_udibono(fecha_val, fecha_venc, tasa_cupon_float,
                                 valor_udi_float, ytm_float, base)
    imprimir_resultado(resultado)


def menu_curvas():
    """Menu para construccion de curvas."""
    print("\n" + "="*60)
    print("CONSTRUCCION DE CURVAS")
    print("="*60)
    print("Se usaran los datos de mercado del proyecto:")
    print()

    datos_mercado = [
        (0.5, 0.0680, 0.0675),
        (1.0, 0.0700, 0.0690),
        (1.5, 0.0720, 0.0705),
        (2.0, 0.0740, 0.0720),
        (2.5, 0.0755, 0.0730),
        (3.0, 0.0770, 0.0745),
    ]

    print("Plazo (anios) | Cupon (%) | YTM (%)")
    print("-" * 40)
    for plazo, cupon, ytm in datos_mercado:
        print("{:13.1f} | {:9.2f} | {:7.2f}".format(plazo, cupon*100, ytm*100))

    input("\nPresione Enter para continuar...")

    resultado = construir_curvas_completas(datos_mercado, frecuencia_cupon=2)
    imprimir_curvas(resultado)


def menu_principal():
    """Menu principal de la calculadora."""
    while True:
        print("\n" + "="*60)
        print("CALCULADORA DE INSTRUMENTOS DE DEUDA")
        print("="*60)
        print("1. Calcular precio de CETE")
        print("2. Calcular M-BONO")
        print("3. Calcular UDIBONO")
        print("4. Construccion de curvas (Bootstrapping y Forwards)")
        print("5. Salir")
        print("="*60)

        opcion = input("Seleccione una opcion: ")

        if opcion == "1":
            menu_cete()
        elif opcion == "2":
            menu_mbono()
        elif opcion == "3":
            menu_udibono()
        elif opcion == "4":
            menu_curvas()
        elif opcion == "5":
            print("\nGracias por usar la calculadora. Hasta luego!\n")
            break
        else:
            print("\nOpcion invalida. Por favor seleccione 1-5.\n")


# Descomentar la siguiente linea para ejecutar el menu interactivo
menu_principal()


CALCULADORA DE INSTRUMENTOS DE DEUDA
1. Calcular precio de CETE
2. Calcular M-BONO
3. Calcular UDIBONO
4. Construccion de curvas (Bootstrapping y Forwards)
5. Salir
Seleccione una opcion: 1

CALCULADORA DE CETE
Fecha de valuacion (YYYY-MM-DD): 2022-05-11
Fecha de vencimiento (YYYY-MM-DD): 2023-05-12
Tasa de descuento (decimal, ej: 0.0675): 0.06
Base de dias (Actual/360 o Actual/365) [Actual/360]: Actual/360

RESULTADOS
instrumento: CETE
fecha_valuacion: 2022-05-11
fecha_vencimiento: 2023-05-12
dias_al_vencimiento: 366
fraccion_anio: 1.016667
tasa_descuento: 0.060000
base_dias: Actual/360
valor_nominal: 10.000000
precio: 9.425071
descuento: 0.574929


CALCULADORA DE INSTRUMENTOS DE DEUDA
1. Calcular precio de CETE
2. Calcular M-BONO
3. Calcular UDIBONO
4. Construccion de curvas (Bootstrapping y Forwards)
5. Salir
Seleccione una opcion: 5

Gracias por usar la calculadora. Hasta luego!



## 7. Demostraciones

### Ejercicio 1: CETE

In [ ]:
print("EJERCICIO 1: CETE")
print("="*60)

resultado_cete = calcular_precio_cete(
    fecha_valuacion="2025-01-15",
    fecha_vencimiento="2025-07-17",
    tasa_descuento=0.0675,
    base_dias="Actual/360"
)

if resultado_cete:
    for clave, valor in resultado_cete.items():
        print("{}: {}".format(clave, valor))
else:
    print("Error en el calculo")

EJERCICIO 1: CETE
instrumento: CETE
fecha_valuacion: 2025-01-15
fecha_vencimiento: 2025-07-17
dias_al_vencimiento: 183
fraccion_anio: 0.508333
tasa_descuento: 0.0675
base_dias: Actual/360
valor_nominal: 10.0
precio: 9.6682579
descuento: 0.3317421


### Prueba: Intentar valuar en dia feriado (1 de enero)

In [ ]:
print("\nPrueba con fecha feriada (1 de enero 2025 - Anio Nuevo):")
print("="*60)

resultado_feriado = calcular_precio_cete(
    fecha_valuacion="2025-01-01",
    fecha_vencimiento="2025-07-17",
    tasa_descuento=0.0675,
    base_dias="Actual/360"
)

if resultado_feriado:
    print("Fecha original: 2025-01-01 (Anio Nuevo)")
    print("Fecha ajustada: {}".format(resultado_feriado['fecha_valuacion']))
    print("\nEl sistema ajusto automaticamente al siguiente dia laboral")
else:
    print("Error en el calculo")


Prueba con fecha feriada (1 de enero 2025 - Anio Nuevo):
Fecha original: 2025-01-01 (Anio Nuevo)
Fecha ajustada: 2025-01-02

El sistema ajusto automaticamente al siguiente dia laboral


### Ejercicio 2: M-BONO

In [ ]:
print("\nEJERCICIO 2: M-BONO")
print("="*60)

resultado_mbono = calcular_mbono(
    fecha_valuacion="2025-01-15",
    fecha_vencimiento="2030-01-15",
    tasa_cupon=0.07,
    ytm=0.0725,
    base_dias="Actual/365"
)

if resultado_mbono:
    for clave, valor in resultado_mbono.items():
        print("{}: {}".format(clave, valor))
else:
    print("Error en el calculo")


EJERCICIO 2: M-BONO
instrumento: M-BONO
fecha_valuacion: 2025-01-15
fecha_vencimiento: 2030-01-15
tasa_cupon: 0.07
ytm: 0.0725
base_dias: Actual/365
valor_nominal: 100.0
cupon_periodico: 3.5
num_cupones_restantes: 10
precio_sucio: 98.955568
interes_devengado: 0.0
precio_limpio: 98.955568
duracion_macaulay: 4.301
duracion_modificada: 4.1505


### Ejercicio 3: UDIBONO

In [ ]:
print("\nEJERCICIO 3: UDIBONO")
print("="*60)

resultado_udibono = calcular_udibono(
    fecha_valuacion="2025-01-15",
    fecha_vencimiento="2035-01-15",
    tasa_cupon_real=0.04,
    valor_udi=8.15,
    ytm_real=0.042,
    base_dias="Actual/365"
)

if resultado_udibono:
    for clave, valor in resultado_udibono.items():
        print("{}: {}".format(clave, valor))
else:
    print("Error en el calculo")


EJERCICIO 3: UDIBONO
instrumento: UDIBONO
fecha_valuacion: 2025-01-15
fecha_vencimiento: 2035-01-15
tasa_cupon_real: 0.04
ytm_real: 0.042
valor_udi: 8.15
base_dias: Actual/365
valor_nominal_udi: 100.0
cupon_periodico_udi: 2.0
num_cupones_restantes: 20
precio_sucio_udi: 98.364927
interes_devengado_udi: 0.0
precio_limpio_udi: 98.364927
precio_sucio_pesos: 801.67
interes_devengado_pesos: 0.0
precio_limpio_pesos: 801.67
duracion_macaulay: 8.3262
duracion_modificada: 8.155


### Ejercicio 4: Construccion de Curvas

In [ ]:
print("\nEJERCICIO 4: CONSTRUCCION DE CURVAS")
print("="*60)

datos_mercado = [
    (0.5, 0.0680, 0.0675),
    (1.0, 0.0700, 0.0690),
    (1.5, 0.0720, 0.0705),
    (2.0, 0.0740, 0.0720),
    (2.5, 0.0755, 0.0730),
    (3.0, 0.0770, 0.0745),
]

resultado_curvas = construir_curvas_completas(datos_mercado, frecuencia_cupon=2)

if resultado_curvas:
    print("\nCurva cuponada original:")
    for punto in resultado_curvas["curva_cupon_original"]:
        print("  Plazo={:.1f}, Cupon={:.4f}, YTM={:.4f}".format(
            punto['plazo'], punto['cupon'], punto['ytm']))

    print("\nCurva cupon cero:")
    for punto in resultado_curvas["curva_cupon_cero"]:
        print("  Plazo={:.1f}, Tasa cero={:.6f}".format(
            punto['plazo'], punto['tasa_cero']))

    print("\nTasas forward:")
    for clave, tasa in resultado_curvas["tasas_forward"].items():
        print("  {}: {:.6f}".format(clave, tasa))
else:
    print("Error en el calculo")


EJERCICIO 4: CONSTRUCCION DE CURVAS

Curva cuponada original:
  Plazo=0.5, Cupon=0.0680, YTM=0.0675
  Plazo=1.0, Cupon=0.0700, YTM=0.0690
  Plazo=1.5, Cupon=0.0720, YTM=0.0705
  Plazo=2.0, Cupon=0.0740, YTM=0.0720
  Plazo=2.5, Cupon=0.0755, YTM=0.0730
  Plazo=3.0, Cupon=0.0770, YTM=0.0745

Curva cupon cero:
  Plazo=0.5, Tasa cero=0.067500
  Plazo=1.0, Tasa cero=0.068826
  Plazo=1.5, Tasa cero=0.070820
  Plazo=2.0, Tasa cero=0.072840
  Plazo=2.5, Tasa cero=0.074368
  Plazo=3.0, Tasa cero=0.075922

Tasas forward:
  1.0y1.0y: 0.076870
  1.0y2.0y: 0.079487
  1.0y3.0y: 0.078297


## Pruebas Adicionales de Validacion

In [ ]:
print("\nPRUEBAS DE VALIDACION")
print("="*60)

# Prueba 1: Fechas incoherentes
print("\n1. Prueba de fechas incoherentes:")
resultado = calcular_precio_cete("2025-07-17", "2025-01-15", 0.0675)
if resultado is None:
    print("   Correctamente rechazado\n")

# Prueba 2: Tasa negativa
print("2. Prueba de tasa negativa:")
resultado = calcular_precio_cete("2025-01-15", "2025-07-17", -0.05)
if resultado is None:
    print("   Correctamente rechazado\n")

# Prueba 3: Tasa muy grande
print("3. Prueba de tasa muy grande (mayor a 1):")
resultado = calcular_precio_cete("2025-01-15", "2025-07-17", 1.5)
if resultado is None:
    print("   Correctamente rechazado\n")

# Prueba 4: Formato de fecha incorrecto
print("4. Prueba de formato de fecha incorrecto:")
resultado = calcular_precio_cete("15-01-2025", "2025-07-17", 0.07)
if resultado is None:
    print("   Correctamente rechazado\n")

# Prueba 5: Dias feriados
print("5. DIAS FERIADOS 2025 (ajuste automatico):")
fechas_prueba = ["2025-01-01", "2025-05-01", "2025-09-16", "2025-12-25"]
for fecha in fechas_prueba:
    res = calcular_precio_cete(fecha, "2025-12-31", 0.07)
    if res:
        print("   {} -> {}".format(fecha, res['fecha_valuacion']))

# Prueba 6: Comparacion de bases de dias
print("\n6. COMPARACION DE BASES DE DIAS:")
for base in ["Actual/360", "Actual/365"]:
    res = calcular_precio_cete("2025-01-15", "2025-07-17", 0.0675, base)
    if res:
        print("   {}: Precio={:.7f}".format(base, res['precio']))

# Prueba 7: Sensibilidad de duracion
print("\n7. SENSIBILIDAD DE DURACION:")
res1 = calcular_mbono("2025-01-15", "2030-01-15", 0.07, 0.07)
res2 = calcular_mbono("2025-01-15", "2030-01-15", 0.07, 0.08)

if res1 and res2:
    cambio_precio = ((res2['precio_limpio'] - res1['precio_limpio']) / res1['precio_limpio']) * 100
    print("   YTM 7% -> 8%: Cambio precio = {:.4f}%".format(cambio_precio))
    print("   Duracion modificada predice: {:.4f}%".format(-res1['duracion_modificada']))

print("\n" + "="*60)


PRUEBAS DE VALIDACION

1. Prueba de fechas incoherentes:
ERROR: Fecha vencimiento debe ser posterior a fecha valuacion
   Correctamente rechazado

2. Prueba de tasa negativa:
ERROR: tasa_descuento debe ser >= 0.0
   Correctamente rechazado

3. Prueba de tasa muy grande (mayor a 1):
ERROR: tasa_descuento debe ser <= 1.0
   Correctamente rechazado

4. Prueba de formato de fecha incorrecto:
ERROR: fecha_valuacion formato invalido. Use YYYY-MM-DD o DD/MM/YYYY
   Correctamente rechazado

5. DIAS FERIADOS 2025 (ajuste automatico):
   2025-01-01 -> 2025-01-02
   2025-05-01 -> 2025-05-02
   2025-09-16 -> 2025-09-17
   2025-12-25 -> 2025-12-26

6. COMPARACION DE BASES DE DIAS:
   Actual/360: Precio=9.6682579
   Actual/365: Precio=9.6726536

7. SENSIBILIDAD DE DURACION:
   YTM 7% -> 8%: Cambio precio = -4.0568%
   Duracion modificada predice: -4.1597%

